# Class Notebook: AIML – Trees, Assumptions & Categorical Features

## Flow for Today
1. Dataset with categorical features and encoding
2. Label encoding vs One-Hot encoding
3. Decision Tree – fit and visualise
4. Overfitting in Decision Trees (depth tuning)
5. Homoscedasticity check with residual plot
6. Multicollinearity check with heatmap and VIF
7. Random Forest – fit and improvement
8. AdaBoost – fit and comparison
9. Model accuracy comparison (DT vs RF vs AdaBoost)
10. Confusion Matrix for best model
11. Feature importance: DT vs RF
12. End summary

## Topic 1: Dataset with Categorical Features

### Q1. How do we create a dataset with categorical features and set up a target?
*Goal: Build a synthetic employee dataset that mixes text categories with numeric columns.*  
*We need this dataset throughout the notebook — encoding, trees, and assumptions all use it.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
n = 300

city       = np.random.choice(['North', 'South', 'East', 'West'], n)
education  = np.random.choice(['HighSchool', 'Graduate', 'Postgraduate'], n)
department = np.random.choice(['Tech', 'Finance', 'HR', 'Marketing'], n)
experience = np.random.randint(0, 20, n)
age        = np.random.randint(22, 60, n)

edu_map  = {'HighSchool': 0, 'Graduate': 1, 'Postgraduate': 2}
city_map = {'North': 1, 'South': 0, 'East': 1, 'West': 0}
dept_map = {'Tech': 2, 'Finance': 1, 'HR': 0, 'Marketing': 1}

score  = (np.array([edu_map[e] for e in education])
         + np.array([city_map[c] for c in city])
         + np.array([dept_map[d] for d in department])
         + experience * 0.15)
target = (score + np.random.normal(0, 1, n) > 3.0).astype(int)

df = pd.DataFrame({
    'city': city, 'education': education, 'department': department,
    'experience': experience, 'age': age, 'salary_high': target
})
print(df.head(10))
print('\nClass balance:\n', df['salary_high'].value_counts())

In [ ]:
# Visual inspection of the dataset
fig, axes = plt.subplots(1, 3, figsize=(13, 3))

df['salary_high'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'])
axes[0].set_title('Target Class Balance')
axes[0].set_xlabel('Salary High (1=Yes)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

df['education'].value_counts().plot(kind='bar', ax=axes[1], color='slateblue')
axes[1].set_title('Education Distribution')
axes[1].tick_params(axis='x', rotation=15)

df['department'].value_counts().plot(kind='bar', ax=axes[2], color='darkorange')
axes[2].set_title('Department Distribution')
axes[2].tick_params(axis='x', rotation=15)

plt.suptitle('Dataset Overview', fontsize=12)
plt.tight_layout()
plt.show()

The three charts give a quick feel for the dataset before any modelling.
Class balance looks reasonable — no extreme imbalance that would skew model training.
We can also see the spread of categorical features like education and department.

We created three categorical columns (city, education, department) plus two numeric ones (age, experience).
The target `salary_high` is derived from a weighted combination of these features.
Keeping data creation in its own cell means we only need to rerun it once.

### Q2. Label encoding vs One-Hot encoding — when to use which?
*Goal: Convert text columns into numbers so that any ML model can read them.*  
*The choice of encoding depends on whether the category has a natural order or not.*

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_enc = df.copy()

# Label encode ordinal: education has a natural order
le = LabelEncoder()
df_enc['education_le'] = le.fit_transform(df_enc['education'])

# One-hot encode nominals: city and department have no order
df_enc = pd.get_dummies(df_enc, columns=['city', 'department'], drop_first=True)
df_enc.drop(columns=['education'], inplace=True)

print('Columns after encoding:', df_enc.columns.tolist())
print(df_enc.head(4))

# Show count of each education label
plt.figure(figsize=(5, 3))
df['education'].value_counts().plot(kind='bar', color='slateblue')
plt.title('Education Category Counts')
plt.xlabel('Education Level')
plt.ylabel('Count')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

Label encoding assigns integers to categories; only use it when there is a natural order (e.g., HighSchool < Graduate < Postgraduate).
One-hot encoding creates binary columns for each category; required for nominals like city or department to avoid false ordinal relationships.
Mixing up the two can silently mislead tree-based and linear models.

## Topic 2: Decision Trees

### Q3a. How do we train a Decision Tree and measure its accuracy?
*Goal: Fit a Decision Tree classifier and see how well it performs on unseen data.*  
*We cap depth at 3 so the tree stays simple — we'll visualise it right after.*

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X = df_enc.drop(columns=['salary_high'])
y = df_enc['salary_high']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)

print('DT Train Accuracy:', round(accuracy_score(y_train, dt.predict(X_train)), 3))
print('DT Test  Accuracy:', round(accuracy_score(y_test,  dt.predict(X_test)),  3))
print('Leaves:', dt.get_n_leaves(), '| Actual depth used:', dt.get_depth())

We trained a Decision Tree with max depth 3 and checked train vs test accuracy.
The leaf and depth counts show how complex the tree actually became.
Next we draw the full tree structure to see exactly what rules it learned.

### Q3b. How does the Decision Tree look visually?
*Goal: Read the actual decision rules the model learned — each box is one split.*  
*Coloured nodes show the dominant class; darker blue = more confident "High salary" prediction.*

In [ ]:
plt.figure(figsize=(14, 5))
plot_tree(dt,
          feature_names=X.columns.tolist(),
          class_names=['Low', 'High'],
          filled=True, rounded=True, fontsize=8)
plt.title('Decision Tree — max_depth=3  (read top to bottom: each node = one question)')
plt.tight_layout()
plt.show()

Each node shows: the split condition, Gini impurity, sample count, and predicted class.
Leaf nodes at the bottom give the final answer — orange = Low salary, blue = High salary.
This is the most powerful way to explain a tree model to students or non-technical audiences.

### Q4. How does tree depth affect overfitting?
*Goal: Show why deeper trees memorise training data but fail on new data.*  
*We sweep depth 1–15 and compare train vs test accuracy to find the optimal depth.*

In [ ]:
depths = range(1, 16)
train_scores, test_scores = [], []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, m.predict(X_train)))
    test_scores.append(accuracy_score(y_test,  m.predict(X_test)))

plt.figure(figsize=(7, 4))
plt.plot(depths, train_scores, 'o-', label='Train', color='steelblue')
plt.plot(depths, test_scores,  's-', label='Test',  color='tomato')
plt.title('Train vs Test Accuracy by Tree Depth')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

Train accuracy keeps rising with depth because the tree memorises training data.
Test accuracy peaks at an optimal depth and then drops — that drop is overfitting.
This chart is a classic tool for choosing a good depth before building ensembles.

## Topic 3: Linear Assumptions — Homoscedasticity & Multicollinearity

### Q5a. How do we fit a regression model and get the residuals?
*Goal: Train a simple linear model so we have predicted values to examine.*  
*Residuals = actual − predicted; we need them to check the homoscedasticity assumption.*

In [ ]:
from sklearn.linear_model import LinearRegression

# Simple regression: predict experience from age + education
df_reg = df_enc[['age', 'education_le', 'experience']].copy()
X_reg  = df_reg[['age', 'education_le']]
y_reg  = df_reg['experience']

lr = LinearRegression()
lr.fit(X_reg, y_reg)
y_pred_reg = lr.predict(X_reg)
residuals  = y_reg - y_pred_reg

print('Coefficient — age:      ', round(lr.coef_[0], 3))
print('Coefficient — education:', round(lr.coef_[1], 3))
print('Intercept:              ', round(lr.intercept_, 2))

# Actual vs Predicted for first 60 samples
plt.figure(figsize=(7, 3))
plt.plot(y_reg.values[:60],  label='Actual',    color='steelblue')
plt.plot(y_pred_reg[:60],    label='Predicted', color='crimson', linestyle='--')
plt.title('Actual vs Predicted Experience (first 60 samples)')
plt.xlabel('Sample Index')
plt.ylabel('Experience (years)')
plt.legend()
plt.tight_layout()
plt.show()

We trained a linear regression model to predict experience from age and education.
The line plot shows how closely the predicted values track the actual ones.
Next we check whether the errors (residuals) are spread evenly — that is homoscedasticity.

### Q5b. What is homoscedasticity and how do we check it?
*Goal: Verify that our model's errors don't grow or shrink as predictions change.*  
*We plot residuals vs fitted values — a flat random band around zero means the assumption holds.*

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(y_pred_reg, residuals, alpha=0.5, color='darkorange')
plt.axhline(0, color='black', linewidth=1.2)
plt.title('Residual Plot — Homoscedasticity Check')
plt.xlabel('Fitted Values (Predicted Experience)')
plt.ylabel('Residuals (Actual − Predicted)')
plt.tight_layout()
plt.show()

A flat, random scatter of dots around zero means homoscedasticity is satisfied.
If dots fan out like a funnel or curve upward, the variance is non-constant — heteroscedasticity.
This check is required before trusting p-values and confidence intervals from a linear model.

### Q6. How do we detect multicollinearity using a heatmap and VIF?
*Goal: Find pairs of features that carry the same information — they mislead linear models.*  
*Heatmap gives a quick visual check; VIF gives a numeric score to decide what to drop.*

In [ ]:
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Add a deliberately correlated feature to show multicollinearity
df_mc = df_enc[['age', 'experience', 'education_le']].copy()
df_mc['age_scaled']  = df_mc['age'] * 0.9 + np.random.normal(0, 1, n)  # correlated with age

# Heatmap
plt.figure(figsize=(5, 4))
sns.heatmap(df_mc.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

# VIF
vif_data = pd.DataFrame()
vif_data['Feature'] = df_mc.columns
vif_data['VIF']     = [variance_inflation_factor(df_mc.values, i) for i in range(df_mc.shape[1])]
print(vif_data)

Multicollinearity occurs when two or more features are highly correlated, making it hard for the model to isolate each feature's effect.
VIF > 5 (or > 10 by stricter rules) flags a problematic feature; here age and age_scaled are nearly identical so one should be dropped.
The heatmap gives a visual overview while VIF gives a quantitative cutoff for removal decisions.

## Topic 4: Ensemble Methods — Random Forest and AdaBoost

### Q7. How does Random Forest improve on a single Decision Tree?
*Goal: See how bagging — training many trees on random subsets — reduces overfitting.*  
*We also plot how accuracy changes as we add more trees to build intuition.*

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

rf_train = accuracy_score(y_train, rf.predict(X_train))
rf_test  = accuracy_score(y_test,  rf.predict(X_test))

print('RF Train Accuracy:', round(rf_train, 3))
print('RF Test  Accuracy:', round(rf_test,  3))

# Accuracy as trees grow
tree_counts = [1, 5, 10, 20, 50, 100]
test_accs   = []
for t in tree_counts:
    m = RandomForestClassifier(n_estimators=t, max_depth=5, random_state=42)
    m.fit(X_train, y_train)
    test_accs.append(accuracy_score(y_test, m.predict(X_test)))

plt.figure(figsize=(6, 4))
plt.plot(tree_counts, test_accs, 'o-', color='seagreen')
plt.title('Random Forest: Test Accuracy vs Number of Trees')
plt.xlabel('n_estimators')
plt.ylabel('Test Accuracy')
plt.tight_layout()
plt.show()

Random Forest builds many trees on random subsets of data and features, then votes on the final class.
This averaging reduces overfitting compared to a single deep tree and generally improves test accuracy.
The chart shows accuracy stabilising as more trees are added — a key reason 100 trees is a common default.

### Q8. How does AdaBoost learn from its mistakes each round?
*Goal: Understand sequential boosting — each new tree focuses on the samples the last one got wrong.*  
*We watch accuracy climb round by round using staged predictions.*

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=42)
ada.fit(X_train, y_train)

ada_train = accuracy_score(y_train, ada.predict(X_train))
ada_test  = accuracy_score(y_test,  ada.predict(X_test))

print('AdaBoost Train Accuracy:', round(ada_train, 3))
print('AdaBoost Test  Accuracy:', round(ada_test,  3))

# Staged accuracy (how AdaBoost improves each round)
staged_acc = [accuracy_score(y_test, p) for p in ada.staged_predict(X_test)]

plt.figure(figsize=(6, 4))
plt.plot(staged_acc, color='darkorange')
plt.title('AdaBoost: Test Accuracy per Boosting Round')
plt.xlabel('Boosting Round')
plt.ylabel('Test Accuracy')
plt.tight_layout()
plt.show()

AdaBoost trains weak learners (shallow trees) sequentially, giving more weight to misclassified samples each round.
The staged accuracy plot shows how the ensemble improves with each added estimator — unlike Random Forest which uses bagging.
Key difference: Random Forest reduces variance; AdaBoost reduces bias by focusing on hard samples.

### Q9. How do DT, RF, and AdaBoost compare side by side?
*Goal: Do a fair model comparison on the same held-out test set.*  
*The bar chart makes it easy to pick the best performer at a glance.*

In [ ]:
dt_best = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_best.fit(X_train, y_train)
dt_test = accuracy_score(y_test, dt_best.predict(X_test))

models  = ['Decision Tree', 'Random Forest', 'AdaBoost']
accs    = [dt_test, rf_test, ada_test]
colors  = ['steelblue', 'seagreen', 'darkorange']

plt.figure(figsize=(6, 4))
plt.bar(models, accs, color=colors)
plt.ylim(0.5, 1.0)
plt.title('Model Comparison: Test Accuracy')
plt.ylabel('Test Accuracy')
for i, v in enumerate(accs):
    plt.text(i, v + 0.005, str(round(v, 3)), ha='center')
plt.tight_layout()
plt.show()

The bar chart compares all three models on the same held-out test set for a fair evaluation.
Ensemble methods (RF, AdaBoost) typically outperform a single Decision Tree on noisy real-world datasets.
Use this as a template for model selection: always compare on the same test split.

### Q10. What does the confusion matrix of the best model tell us?
*Goal: Go beyond accuracy and see exactly where the model makes mistakes.*  
*TN, FP, FN, TP each reveal a different kind of error that a single number hides.*

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

best_pred = rf.predict(X_test)
cm = confusion_matrix(y_test, best_pred)
tn, fp, fn, tp = cm.ravel()

print(classification_report(y_test, best_pred, target_names=['Low Salary', 'High Salary']))

plt.figure(figsize=(4, 3))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix – Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')
labels = [['TN', 'FP'], ['FN', 'TP']]
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{labels[i][j]}\n{cm[i,j]}", ha='center', va='center', fontsize=11)
plt.tight_layout()
plt.show()

The confusion matrix breaks accuracy into four buckets: TN, FP, FN, TP — essential for understanding class-level performance.
Precision and recall in the report reveal trade-offs that a single accuracy number hides.
For salary prediction, false negatives (predicting Low when actually High) may be more costly to a recruiter than false positives.

### Q11. Which features matter most — and do DT and RF agree?
*Goal: Rank features by importance to understand what the model relies on most.*  
*Comparing both models shows whether feature rankings are stable or depend on the algorithm.*

In [ ]:
feat_names = X.columns.tolist()

dt_imp = pd.Series(dt_best.feature_importances_, index=feat_names).sort_values(ascending=False)
rf_imp = pd.Series(rf.feature_importances_,      index=feat_names).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
dt_imp.head(8).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Feature Importance – Decision Tree')
axes[0].invert_yaxis()

rf_imp.head(8).plot(kind='barh', ax=axes[1], color='seagreen')
axes[1].set_title('Feature Importance – Random Forest')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

Feature importances show which columns the model relied on most to make splits.
Single DT importance can be noisy; RF averages across 100 trees giving a more stable ranking.
Comparing both charts reveals if a feature consistently ranks high — those are the key drivers.

## End Summary
- Categorical features: label-encode ordinal, one-hot encode nominal
- Decision Tree: great visual tool but prone to overfitting without depth control
- Homoscedasticity: residuals should show no pattern around zero
- Multicollinearity: high correlation + VIF > 5 → drop one of the correlated features
- Random Forest: reduces variance through bagging and feature randomness
- AdaBoost: reduces bias by sequentially fixing mistakes of previous learners
- Always compare models on the same test split; use confusion matrix, not just accuracy